# Lab 2: RAG Pipeline Evaluation — Naive vs Metadata-Aware

**AI-2: AI Backend Engineering** | Northbrook Partners Document QA

---

**Weight:** 20% of course grade
**Due:** Start of Session 4.2
**Estimated time:** 2-3 hours

## Scenario

In Session 3.1, you built a naive RAG pipeline. In Session 3.2, you explored metadata filtering and saw that it can dramatically change retrieval results. But *when* does filtering actually help? And *how much* does it help?

Your job: design an evaluation, run both pipelines head-to-head, and analyze the results.

## Rubric

| Section | Points | What I'm Looking For |
|---------|--------|---------------------|
| Evaluation Runner | 25 | `run_evaluation()` works correctly, computes meaningful metrics |
| Query Design | 25 | 7+ diverse queries, good type coverage, thoughtful reasoning |
| Summary & Analysis Function | 25 | `summarize_evaluation()` correct, identifies wins/losses |
| Written Analysis | 25 | Q1-Q3 show genuine understanding of retrieval tradeoffs |
| **Total** | **100** | **Passing: 70+** |

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
load_dotenv()

from src.s4_retrieval.naive import naive_retrieve
from src.s4_retrieval.filtered import filtered_retrieve, compare_retrieval
from src.s3_ingestion.store import get_collection

print("Setup complete.")

In [ ]:
collection = get_collection()
sample = collection.peek(limit=3)
print(f"Collection has {collection.count()} chunks")
print(f"\nMetadata fields: {list(sample['metadatas'][0].keys())}")
print(f"\nDocument types in corpus:")
all_meta = collection.get(include=["metadatas"])["metadatas"]
doc_types = {}
for m in all_meta:
    dt = m.get("doc_type", "unknown")
    doc_types[dt] = doc_types.get(dt, 0) + 1
for dt, count in sorted(doc_types.items()):
    print(f"  {dt}: {count} chunks")

## Section 1: Design Your Evaluation Queries (25 pts)

A good evaluation needs diverse queries that test different aspects of your pipeline. Design **7 or more** queries with these requirements:

| Requirement | Minimum |
|-------------|--------|
| Policy questions | 2+ |
| Financial questions | 1+ |
| Meeting or memo questions | 1+ |
| Cross-document or ambiguous questions | 1+ |
| Edge cases (over-filtering risk) | 1+ |

For each query, specify:
- `question`: The question text
- `expected_type`: Which `doc_type` should contain the answer
- `reasoning`: Why you chose this query and what it tests (1 sentence)

In [ ]:
EVAL_QUERIES = [
    # -- Provided Starters (3 queries) ------------------------------------
    {
        "question": "What equipment stipend does Northbrook provide for remote workers?",
        "expected_type": "policy",
        "reasoning": "Direct policy lookup — answer is concentrated in remote_work_policy.md",
    },
    {
        "question": "What was Northbrook's Q4 2024 revenue?",
        "expected_type": "financial",
        "reasoning": "Financial fact retrieval — tests whether filtering to financial docs helps",
    },
    {
        "question": "What did the CEO announce at the annual kickoff?",
        "expected_type": "memo",
        "reasoning": "Memo retrieval — answer spans multiple topics in one document",
    },

    # -- YOUR QUERIES BELOW (add 4+ more to reach 7 total) ----------------
    # Include at least:
    #   - 1 more policy question
    #   - 1 meeting question
    #   - 1 cross-document question (answer may span multiple doc_types)
    #   - 1 edge case (where filtering might NOT help or might hurt)
    #
    # Format:
    # {
    #     "question": "...",
    #     "expected_type": "policy" | "financial" | "meeting" | "memo" | None,
    #     "reasoning": "...",
    # },

    # YOUR CODE HERE

]

print(f"Total queries: {len(EVAL_QUERIES)}")
for i, q in enumerate(EVAL_QUERIES):
    print(f"  {i+1}. [{q['expected_type']}] {q['question'][:70]}")

## Section 2: Build Your Evaluation Runner (25 pts)

Write a function that runs both pipelines on every query and computes comparison metrics.

For each query, you need to:
1. Build the ChromaDB `where` clause from `expected_type`
2. Run `compare_retrieval()` to get both pipelines' results
3. Compute **source precision** for each pipeline (fraction of retrieved sources matching `expected_type`)
4. Determine a **verdict**: did filtering help, hurt, or make no difference?

**Functions you'll use:**
- `compare_retrieval(question, filters, top_k)` → dict with keys: `naive_sources`, `filtered_sources`, `naive_top_score`, `filtered_top_score`, `overlap`
- `naive_retrieve(question, top_k)` → list of source dicts (each has `text`, `metadata`, `score`)

**Key detail:** When `expected_type` is `None`, skip filtering — just run `naive_retrieve()` and set verdict to `"no_filter"`.

In [ ]:
def run_evaluation(queries: list[dict], top_k: int = 5) -> list[dict]:
    """Run naive and filtered retrieval on every query and compute metrics.

    Args:
        queries: List of dicts with "question", "expected_type", "reasoning"
        top_k: Number of chunks to retrieve per pipeline

    Returns:
        List of dicts, each containing:
            - question (str)
            - expected_type (str)
            - filter_used (dict): the ChromaDB where clause
            - naive_sources (list): results from naive pipeline
            - filtered_sources (list): results from filtered pipeline
            - naive_precision (float): fraction of naive sources matching expected_type
            - filtered_precision (float): fraction of filtered sources matching expected_type
            - naive_top_score (float): best similarity score from naive
            - filtered_top_score (float): best similarity score from filtered
            - overlap (int): chunks appearing in both result sets
            - verdict (str): "filtered_better" | "naive_better" | "tie"
    """
    results = []

    for q in queries:
        # -- Step 1 ---------------------------------------------------
        # Build the filter dict from expected_type.
        # If expected_type is None, skip filtering for this query.
        # Example: expected_type="policy" -> {"doc_type": "policy"}

        # YOUR CODE HERE


        # -- Step 2 ---------------------------------------------------
        # Run compare_retrieval() to get both pipelines' results.
        # If expected_type is None, run only naive_retrieve() and
        # set filtered_sources to an empty list.
        # Hint: compare_retrieval(question, filters, top_k)

        # YOUR CODE HERE


        # -- Step 3 ---------------------------------------------------
        # Compute source precision for BOTH pipelines.
        # Precision = (number of sources with matching doc_type) / (total sources)
        # For naive: count how many naive_sources have metadata["doc_type"] == expected_type
        # For filtered: same calculation on filtered_sources
        # Handle empty source lists (precision = 0.0)

        # YOUR CODE HERE


        # -- Step 4 ---------------------------------------------------
        # Determine the verdict.
        # If filtered_precision > naive_precision -> "filtered_better"
        # If naive_precision > filtered_precision -> "naive_better"
        # If equal -> "tie"
        # If expected_type is None -> "no_filter"

        # YOUR CODE HERE


        # -- Step 5 ---------------------------------------------------
        # Build the result dict and append to results.

        # YOUR CODE HERE

    return results

In [ ]:
results = run_evaluation(EVAL_QUERIES, top_k=5)

print(f"{'#':<3} {'Query':<55} {'Expected':<12} {'N-Prec':>7} {'F-Prec':>7} {'Verdict':<16}")
print("-" * 105)
for i, r in enumerate(results):
    print(f"{i+1:<3} {r['question'][:53]:<55} {r['expected_type'] or 'any':<12} "
          f"{r['naive_precision']:>6.0%} {r['filtered_precision']:>6.0%}  {r['verdict']}")

In [ ]:
for i, r in enumerate(results):
    print(f"\n{'='*80}")
    print(f"Query {i+1}: {r['question']}")
    print(f"Expected type: {r['expected_type']}  |  Filter: {r.get('filter_used', 'None')}")
    print(f"Verdict: {r['verdict']}")
    print(f"\n  Naive sources ({r['naive_precision']:.0%} precision, top score: {r['naive_top_score']:.3f}):")
    for s in r.get('naive_sources', [])[:5]:
        doc_type = s['metadata'].get('doc_type', '?')
        match = 'Y' if doc_type == r['expected_type'] else 'N'
        print(f"    {match} [{s['score']:.3f}] {s['metadata']['source']} ({doc_type})")
    print(f"\n  Filtered sources ({r['filtered_precision']:.0%} precision, top score: {r['filtered_top_score']:.3f}):")
    for s in r.get('filtered_sources', [])[:5]:
        doc_type = s['metadata'].get('doc_type', '?')
        match = 'Y' if doc_type == r['expected_type'] else 'N'
        print(f"    {match} [{s['score']:.3f}] {s['metadata']['source']} ({doc_type})")

## Section 3: Summarize Your Evaluation (25 pts)

Now compute overall metrics. How often did filtering win? What was the average precision improvement? Which query showed the biggest win for filtering, and which showed the biggest loss?

In [ ]:
def summarize_evaluation(results: list[dict]) -> dict:
    """Compute overall evaluation metrics from the comparison results.

    Args:
        results: Output from run_evaluation()

    Returns:
        Dict with keys:
            - total_queries (int)
            - filtered_wins (int): queries where filtering was better
            - naive_wins (int): queries where naive was better
            - ties (int): queries where both were equal
            - avg_naive_precision (float): mean precision across all queries
            - avg_filtered_precision (float): mean precision across all queries
            - precision_lift (float): avg_filtered - avg_naive
            - best_win (dict or None): the result dict where filtering helped MOST
              (biggest positive difference in precision)
            - worst_loss (dict or None): the result dict where filtering hurt MOST
              (biggest negative difference in precision)
    """
    # -- Step 1 -------------------------------------------------------
    # Count verdicts: how many filtered_better, naive_better, tie?

    # YOUR CODE HERE


    # -- Step 2 -------------------------------------------------------
    # Calculate average precision for both pipelines.
    # Hint: sum(r["naive_precision"] for r in results) / len(results)

    # YOUR CODE HERE


    # -- Step 3 -------------------------------------------------------
    # Find best_win: the result with the largest POSITIVE
    # (filtered_precision - naive_precision) difference.
    # Find worst_loss: the result with the largest NEGATIVE difference.
    # If no wins, best_win = None. If no losses, worst_loss = None.

    # YOUR CODE HERE


    return {
        "total_queries": ...,
        "filtered_wins": ...,
        "naive_wins": ...,
        "ties": ...,
        "avg_naive_precision": ...,
        "avg_filtered_precision": ...,
        "precision_lift": ...,
        "best_win": ...,
        "worst_loss": ...,
    }

In [ ]:
summary = summarize_evaluation(results)

print("EVALUATION SUMMARY")
print("=" * 50)
print(f"Total queries:          {summary['total_queries']}")
print(f"Filtering wins:         {summary['filtered_wins']}")
print(f"Naive wins:             {summary['naive_wins']}")
print(f"Ties:                   {summary['ties']}")
print(f"")
print(f"Avg naive precision:    {summary['avg_naive_precision']:.1%}")
print(f"Avg filtered precision: {summary['avg_filtered_precision']:.1%}")
print(f"Precision lift:         {summary['precision_lift']:+.1%}")

if summary["best_win"]:
    bw = summary["best_win"]
    print(f"\nBiggest filtering win:")
    print(f"  Query: {bw['question']}")
    print(f"  Naive precision: {bw['naive_precision']:.0%} -> Filtered: {bw['filtered_precision']:.0%}")

if summary["worst_loss"]:
    wl = summary["worst_loss"]
    print(f"\nBiggest filtering loss:")
    print(f"  Query: {wl['question']}")
    print(f"  Naive precision: {wl['naive_precision']:.0%} -> Filtered: {wl['filtered_precision']:.0%}")

## Section 4: Written Analysis (25 pts)

Answer the following questions in the markdown cells below. Your analysis should demonstrate that you understand *why* filtering helped or hurt, not just *that* it did.

### Q1: Your Best Filtering Win (10 pts)

Pick the query where filtering improved results the most. Answer these questions in 5-7 sentences:
- What did naive retrieval get wrong? (What irrelevant sources did it include?)
- What did the filter fix? (How did restricting by doc_type improve the results?)
- Why does this query benefit from filtering? (What about the question makes doc_type filtering effective?)

*Your answer here*

### Q2: When Filtering Didn't Help (8 pts)

Pick a query where filtering made no difference or actually hurt results. Answer in 3-5 sentences:
- What happened? (Did both pipelines return the same results? Did filtering remove relevant chunks?)
- Why didn't filtering help for this query? (Is the answer spread across doc types? Is the question ambiguous?)

*Your answer here*

### Q3: A Rule for Production (7 pts)

Based on your evaluation results, propose a decision rule for when a production system should use filtered vs naive retrieval. Your rule should be specific enough that a developer could implement it. (3-5 sentences)

Consider: What signals in the question text indicate that filtering will help? When should the system fall back to naive? How would you handle ambiguous cases?

*Your answer here*

## Section 5: Observability (After Session 4.1) — Optional Bonus

After Session 4.1, come back and add PipelineLogger integration to your evaluation runner. This is **optional** — your lab grade is based on Sections 1-4 above.

*This section will be covered in Session 4.1. Leave it for now.*

In [ ]:
# After Session 4.1, uncomment and integrate PipelineLogger:
#
# from src.s5_observability.logger import PipelineLogger
#
# logger = PipelineLogger("lab2_evaluation")
#
# Add logging calls inside run_evaluation():
#   logger.log_retrieval(question, sources, scores, latency)
#   logger.log_generation(question, answer, input_tokens, output_tokens)
#
# Then display the log summary:
#   logger.summary()

print("PipelineLogger integration — complete after Session 4.1")

## Before You Submit

- [ ] `EVAL_QUERIES` has 7+ queries with diverse types and reasoning
- [ ] `run_evaluation()` runs without errors on all queries
- [ ] `summarize_evaluation()` produces correct counts and identifies best win/worst loss
- [ ] Q1 explains a specific filtering win with detail
- [ ] Q2 explains a case where filtering didn't help
- [ ] Q3 proposes a concrete decision rule
- [ ] (Optional) PipelineLogger integrated after Session 4.1